In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/notebooks/shahabahmad123/lama7b-tuning-newdata/__results__.html
/kaggle/input/notebooks/shahabahmad123/lama7b-tuning-newdata/__notebook__.ipynb
/kaggle/input/notebooks/shahabahmad123/lama7b-tuning-newdata/__output__.json
/kaggle/input/notebooks/shahabahmad123/lama7b-tuning-newdata/custom.css
/kaggle/input/notebooks/shahabahmad123/lama7b-tuning-newdata/checkpoints/dolly_lora/README.md
/kaggle/input/notebooks/shahabahmad123/lama7b-tuning-newdata/checkpoints/dolly_lora/checkpoint-470/adapter_model.safetensors
/kaggle/input/notebooks/shahabahmad123/lama7b-tuning-newdata/checkpoints/dolly_lora/checkpoint-470/trainer_state.json
/kaggle/input/notebooks/shahabahmad123/lama7b-tuning-newdata/checkpoints/dolly_lora/checkpoint-470/training_args.bin
/kaggle/input/notebooks/shahabahmad123/lama7b-tuning-newdata/checkpoints/dolly_lora/checkpoint-470/adapter_config.json
/kaggle/input/notebooks/shahabahmad123/lama7b-tuning-newdata/checkpoints/dolly_lora/checkpoint-470/README.md
/kaggle/inp

In [3]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 12.1 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.2 MB/s eta 0:00:00:00:0100:01


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import pandas as pd

# --- EXACT PATHS FROM YOUR KAGGLE ENVIRONMENT ---
MODEL_PATH = '/kaggle/input/models/metaresearch/llama-2/pytorch/7b-hf/1' 
PEFT_PATH  = '/kaggle/input/notebooks/shahabahmad123/lama7b-tuning-newdata/checkpoints/dolly_lora_epoch1' 
DATA_PATH  = '/kaggle/input/notebooks/shahabahmad123/lama7b-tuning-newdata/datasets/dolly/dolly_data.csv'
NEW_CKPT_PATH = '/kaggle/working/checkpoints/dolly_lora_epoch2'

print("1. Loading base LLaMA-7B...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("2. Attaching Epoch 1 LoRA weights...")
# is_trainable=True is CRITICAL to continue training
model = PeftModel.from_pretrained(
    base_model, 
    PEFT_PATH, 
    is_trainable=True 
)
model.print_trainable_parameters()

print("3. Loading Dolly dataset...")
dolly_df = pd.read_csv(DATA_PATH)
dolly_dataset = Dataset.from_pandas(dolly_df[['text']])

print("4. Configuring Trainer for Epoch 2...")
training_args = SFTConfig(
    output_dir=f'{NEW_CKPT_PATH}_logs',
    num_train_epochs=1,                  # 1 new epoch (Epoch 2 total)
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=3e-4,                  
    warmup_steps=0,                      # No warmup needed, done in Epoch 1
    lr_scheduler_type="linear",
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
    report_to="none",
    dataloader_num_workers=2,
    dataset_text_field="text",
    max_length=2048,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dolly_dataset,
    args=training_args,
)

print("5. Starting Dolly LoRA training — Epoch 2...")
trainer.train()
print("Epoch 2 complete.")

# Save Epoch 2 to /kaggle/working/
model.save_pretrained(NEW_CKPT_PATH)
tokenizer.save_pretrained(NEW_CKPT_PATH)
print(f"✅ Epoch 2 saved to: {NEW_CKPT_PATH}")

1. Loading base LLaMA-7B...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


2. Attaching Epoch 1 LoRA weights...
trainable params: 4,194,304 || all params: 6,742,609,920 || trainable%: 0.0622
3. Loading Dolly dataset...
4. Configuring Trainer for Epoch 2...


...

Tokenizing train dataset:   0%|          | 0/15011 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


5. Starting Dolly LoRA training — Epoch 2...


Step,Training Loss
50,1.364193
100,1.352403
150,1.318484
200,1.292461
250,1.317913
300,1.313311
350,1.296145
400,1.324274
450,1.304190


Epoch 2 complete.
✅ Epoch 2 saved to: /kaggle/working/checkpoints/dolly_lora_epoch2
